# Weakly-Supervised Bone Segmentation — FracAtlas
### Pipeline: DenseNet121 → LayerCAM → SAM ViT-B → U-Net

**Stages:**
1. Setup & Dataset Download
2. Stage 1 — Anatomy Classifier Training
3. Stage 2 — Pseudo Mask Generation (LayerCAM + SAM)
4. Stage 3 — U-Net Segmentation Training
5. Inference & Visualization
6. Debug & Experiments
7. Bottleneck Analysis (100 ảnh)
8. Multi-Mask Fusion
9. Best Strategy Training
10. Kiểm chứng giả thuyết — Ranking chọn sai?

## 1. Setup

In [ ]:
# Mount Google Drive — outputs will be saved here across sessions
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (skip if already cloned)
import os
if not os.path.exists('/content/Thesis'):
    !git clone https://github.com/itsthang333/Thesis.git /content/Thesis
else:
    print('Repo already exists, pulling latest...')
    !git -C /content/Thesis pull

%cd /content/Thesis/project
!pwd

In [ ]:
# Install dependencies
!pip install -q torch torchvision numpy pillow tqdm opencv-python kagglehub
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
print('All dependencies installed.')

In [ ]:
# Download FracAtlas dataset from Kaggle
import kagglehub

DATASET_PATH = kagglehub.dataset_download('mahmudulhasantasin/fracatlas-original-dataset')
print(f'Dataset path: {DATASET_PATH}')

# Verify structure
import os
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    if level < 2:
        print('  ' * level + os.path.basename(root) + '/')
    if level == 1:
        print('  ' * (level + 1) + f'({len(files)} files)')

In [ ]:
# Download SAM ViT-B checkpoint (~375 MB)
# Saved to Drive so it persists across Colab sessions
import os, urllib.request

SAM_CHECKPOINT_DIR = '/content/drive/MyDrive/ThesisOutputs/checkpoints'
SAM_CHECKPOINT_PATH = os.path.join(SAM_CHECKPOINT_DIR, 'sam_vit_b_01ec64.pth')
SAM_DOWNLOAD_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

os.makedirs(SAM_CHECKPOINT_DIR, exist_ok=True)

if os.path.exists(SAM_CHECKPOINT_PATH):
    print(f'SAM checkpoint already exists: {SAM_CHECKPOINT_PATH}')
else:
    print('Downloading SAM ViT-B checkpoint (~375 MB)...')
    urllib.request.urlretrieve(SAM_DOWNLOAD_URL, SAM_CHECKPOINT_PATH)
    print(f'Saved to {SAM_CHECKPOINT_PATH}')

## 2. Stage 1 — Anatomy Classifier Training

Train DenseNet121 with BCEWithLogitsLoss on 4 anatomy labels: `hand, leg, hip, shoulder`.

In [ ]:
CLASSIFIER_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/classifier'

!python train_classifier.py \
  --data-root "{DATASET_PATH}" \
  --target-columns hand,leg,hip,shoulder \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 2 \
  --epochs 25 \
  --output-dir "{CLASSIFIER_OUTPUT}"

## 3. Stage 2 — Pseudo Mask Generation (LayerCAM + SAM)

Pipeline:
- **LayerCAM** on denseblock2/3/4 → confidence-filtered weighted fusion
- **Adaptive threshold** (percentile 85) → peak prompts
- **SAM ViT-B** generates candidate masks from prompts
- **CAM-guided selection** (mean CAM score ≥ 0.4) + logical-OR fusion
- **Morphological refinement** (closing → opening → fill_holes → remove_small)

In [ ]:
PSEUDO_MASK_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/pseudo_masks'
CLASSIFIER_CHECKPOINT = f'{CLASSIFIER_OUTPUT}/best_classifier.pt'

# False: preview nhanh 10 ảnh. True: tạo toàn bộ pseudo masks trước khi train U-Net.
RUN_FULL_PSEUDO_MASKS = False
PROCESS_MODE = '--process-all' if RUN_FULL_PSEUDO_MASKS else '--max-images 10'

# Cấu hình mới: CAM chọn full bone components, SAM nhận bbox + structured points.
MORPHOLOGY_FUSION_MODE = 'components'  # ablation cũ: 'weighted'
SAM_PROMPT_MODE = 'box_point'          # point | joint_points | box | box_point

!python generate_pseudo_masks.py \
  --data-root "{DATASET_PATH}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --target-columns hand,leg,hip,shoulder \
  --image-size 384 \
  --batch-size 1 \
  --num-workers 2 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --selection-method bone_hybrid \
  --fusion-topk 3 \
  --morphology-fusion-mode "{MORPHOLOGY_FUSION_MODE}" \
  --sam-prompt-mode "{SAM_PROMPT_MODE}" \
  --max-bone-components 6 \
  --points-per-component 3 \
  --bbox-padding-ratio 0.05 \
  --negative-points-per-component 0 \
  --bone-seed-percentile 88 \
  --bone-support-percentile 62 \
  --closing-kernel 5 \
  --opening-kernel 0 \
  --max-hole-area 500 \
  --min-size 40 \
  --save-visuals-limit 10 \
  {PROCESS_MODE} \
  --output-dir "{PSEUDO_MASK_OUTPUT}"

In [ ]:
# Sanity check: count generated masks and show a sample
import glob, os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

mask_files = glob.glob(f'{PSEUDO_MASK_OUTPUT}/masks/*.png')
print(f'Generated {len(mask_files)} pseudo masks')

if mask_files:
    sample_path = mask_files[0]
    stem = os.path.splitext(os.path.basename(sample_path))[0]
    overlay_path = f'{PSEUDO_MASK_OUTPUT}/overlays/{stem}_fused_layercam.png'

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(np.array(Image.open(sample_path)), cmap='gray')
    axes[0].set_title('Pseudo Mask')
    axes[0].axis('off')

    if os.path.exists(overlay_path):
        axes[1].imshow(np.array(Image.open(overlay_path)))
        axes[1].set_title('Fused LayerCAM Overlay')
        axes[1].axis('off')

    plt.suptitle(f'Sample: {stem}', fontsize=11)
    plt.tight_layout()
    plt.show()

## 4. Stage 3 — U-Net Segmentation Training

Train U-Net (encoder 64→1024) with `0.5 × BCE + 0.5 × Dice` loss on the pseudo masks.

In [ ]:
SEGMENTATION_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/segmentation'

import glob
pseudo_mask_count = len(glob.glob(f'{PSEUDO_MASK_OUTPUT}/masks/*.png'))
assert pseudo_mask_count > 1000, (
    f'Chỉ tìm thấy {pseudo_mask_count} pseudo masks. '
    'Hãy đặt RUN_FULL_PSEUDO_MASKS=True và chạy lại Stage 2 trước khi train.'
)
print(f'Training with {pseudo_mask_count} pseudo masks')

!python train_segmentation.py \
  --data-root "{DATASET_PATH}" \
  --mask-root "{PSEUDO_MASK_OUTPUT}/masks" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 2 \
  --epochs 25 \
  --output-dir "{SEGMENTATION_OUTPUT}"

In [ ]:
# Plot training curves
import pandas as pd
import matplotlib.pyplot as plt
import os

log_path = f'{SEGMENTATION_OUTPUT}/training_log.csv'
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for ax, metric in zip(axes, ['loss', 'dice', 'iou']):
        ax.plot(df['epoch'], df[f'train_{metric}'], label='train')
        ax.plot(df['epoch'], df[f'val_{metric}'], label='val')
        ax.set_title(metric.capitalize())
        ax.set_xlabel('Epoch')
        ax.legend()

    plt.suptitle('U-Net Training Curves', fontsize=13)
    plt.tight_layout()
    plt.show()

    best = df.loc[df['val_dice'].idxmax()]
    print(f"Best epoch: {int(best['epoch'])} | "
          f"Val Dice: {best['val_dice']:.4f} | "
          f"Val IoU:  {best['val_iou']:.4f}")

## 5. Inference & Visualization

In [ ]:
# Pick a random test image from the dataset
import glob, random, os

all_images = (
    glob.glob(f'{DATASET_PATH}/**/*.jpg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.jpeg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.png', recursive=True)
)
TEST_IMAGE = random.choice(all_images)
print(f'Test image: {TEST_IMAGE}')

INFERENCE_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/inference'

!python inference.py \
  --image-path "{TEST_IMAGE}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --segmentation-checkpoint "{SEGMENTATION_OUTPUT}/best_unet.pt" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --image-size 384 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --selection-method bone_hybrid \
  --fusion-topk 3 \
  --bone-seed-percentile 88 \
  --bone-support-percentile 62 \
  --closing-kernel 5 \
  --opening-kernel 0 \
  --max-hole-area 500 \
  --min-size 40 \
  --output-dir "{INFERENCE_OUTPUT}"

In [ ]:
# Visualize full pipeline output
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

stem = os.path.splitext(os.path.basename(TEST_IMAGE))[0]
panels = {
    'Original':           TEST_IMAGE,
    'Fused LayerCAM':     f'{INFERENCE_OUTPUT}/{stem}_fused_layercam.png',
    'Pseudo Mask (SAM)':  f'{INFERENCE_OUTPUT}/{stem}_pseudo_mask.png',
    'Segmentation Mask':  f'{INFERENCE_OUTPUT}/{stem}_segmentation_mask.png',
    'Final Overlay':      f'{INFERENCE_OUTPUT}/{stem}_final_overlay.png',
}

missing = [title for title, path in panels.items() if not os.path.exists(path)]
if missing:
    print(f'WARNING: missing output files: {missing}')

fig, axes = plt.subplots(1, len(panels), figsize=(20, 4))
for ax, (title, path) in zip(axes, panels.items()):
    if os.path.exists(path):
        ax.imshow(np.array(Image.open(path).convert('RGB')))
    else:
        ax.text(0.5, 0.5, 'Not found', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.suptitle(f'Pipeline Output — {stem}', fontsize=11)
plt.tight_layout()
plt.show()

---
## 6. Debug & Experiments

> Chạy sau khi có `best_classifier.pt`. Không cần train U-Net xong.
>
> Mục tiêu: xác định bottleneck — CAM / Prompt / SAM / Selection — trước khi train full pipeline.

### Experiment Map
| # | Câu hỏi | File xem |
|---|---------|----------|
| E1 | SAM có sinh mask bao phủ toàn bộ xương không? | `debug/<stem>/mask_*.png`, `scores.json` |
| E2 | LayerCAM có cover đủ xương không? | `debug/<stem>/layercam_with_points.png` |
| E3 | Prompt có rơi đúng trên xương không? | `debug/<stem>/foreground.png`, `component_*.png` |
| E4 | Selection method nào tốt nhất? | so sánh `mean` / `sum` / `mean_area` |
| E5 | Batch 20 ảnh đại diện | pipeline strip cho 5 ảnh/class |

In [ ]:
# ── E1 + E2 + E3: Run visualize_pipeline.py with --debug on one image ────────
# Prerequisite: CLASSIFIER_CHECKPOINT and SAM_CHECKPOINT_PATH must be defined above.

import glob, random, os

all_images = (
    glob.glob(f'{DATASET_PATH}/**/*.jpg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.jpeg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.png', recursive=True)
)
DEBUG_IMAGE = random.choice(all_images)
print(f'Debug image: {DEBUG_IMAGE}')

DEBUG_VIZ_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/debug_viz'

!python visualize_pipeline.py \
  --image-path "{DEBUG_IMAGE}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --image-size 384 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --selection-method mean \
  --debug \
  --output-path "{DEBUG_VIZ_OUTPUT}/pipeline_strip.png"

In [ ]:
# ── E1 + E2 + E3: Display pipeline strip + all debug outputs ─────────────────
import os, json, glob
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

stem = os.path.splitext(os.path.basename(DEBUG_IMAGE))[0]
# debug dir is <output-path parent>/debug/<stem>/ — output-path was DEBUG_VIZ_OUTPUT/pipeline_strip.png
debug_dir_viz = f'{DEBUG_VIZ_OUTPUT}/debug/{stem}'
viz_strip = f'{DEBUG_VIZ_OUTPUT}/pipeline_strip.png'

# --- Panel 1: Pipeline strip ---
if os.path.exists(viz_strip):
    fig, ax = plt.subplots(figsize=(22, 3.5))
    ax.imshow(np.array(Image.open(viz_strip)))
    ax.axis('off')
    ax.set_title('Pipeline Strip', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print(f'WARNING: pipeline strip not found at {viz_strip}')

# --- Panel 2: All SAM candidate masks ---
mask_files = sorted(glob.glob(f'{debug_dir_viz}/mask_*.png'))
overlay_files = sorted(glob.glob(f'{debug_dir_viz}/overlay_mask_*.png'))

if mask_files:
    n = len(mask_files)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    if n == 1:
        axes = [[axes[0]], [axes[1]]]
    for i, (mf, of) in enumerate(zip(mask_files, overlay_files)):
        axes[0][i].imshow(np.array(Image.open(mf)), cmap='gray')
        axes[0][i].set_title(f'mask_{i}', fontsize=9)
        axes[0][i].axis('off')
        if os.path.exists(of):
            axes[1][i].imshow(np.array(Image.open(of).convert('RGB')))
        axes[1][i].set_title(f'overlay_{i}', fontsize=9)
        axes[1][i].axis('off')
    plt.suptitle(f'E1 — SAM Candidate Masks: {stem}', fontsize=11)
    plt.tight_layout()
    plt.show()

    scores_path = f'{debug_dir_viz}/scores.json'
    if os.path.exists(scores_path):
        with open(scores_path) as f:
            scores = json.load(f)
        print('scores.json:')
        for k, v in scores.items():
            bar = '█' * int(v['score'] * 30)
            print(f"  {k}: score={v['score']:.3f}  area={v['area']:6d}  {bar}")
else:
    print(f'No debug masks found. Expected: {debug_dir_viz}')
    print('Make sure --debug flag was passed to visualize_pipeline.py above.')

# --- Panel 3: Prompt + foreground debug ---
fg_path  = f'{debug_dir_viz}/foreground.png'
pts_path = f'{debug_dir_viz}/layercam_with_points.png'
comp_files = sorted(glob.glob(f'{debug_dir_viz}/component_*.png'))

debug_panels = []
if os.path.exists(fg_path):  debug_panels.append(('Foreground (E3)', fg_path))
if os.path.exists(pts_path): debug_panels.append(('Prompts on CAM (E2)', pts_path))
for cf in comp_files[:4]:
    debug_panels.append((os.path.basename(cf), cf))

if debug_panels:
    fig, axes = plt.subplots(1, len(debug_panels), figsize=(5 * len(debug_panels), 4))
    if len(debug_panels) == 1: axes = [axes]
    for ax, (title, path) in zip(axes, debug_panels):
        ax.imshow(np.array(Image.open(path).convert('RGB')))
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    plt.suptitle(f'E2 + E3 — CAM Coverage & Prompts: {stem}', fontsize=11)
    plt.tight_layout()
    plt.show()

### E4 — So sánh Selection Method (`mean` / `sum` / `mean_area`)

Chạy `generate_pseudo_masks.py` 3 lần với selection method khác nhau trên cùng tập ảnh nhỏ, rồi so sánh pseudo mask bằng mắt và Dice/IoU (nếu có GT).

In [ ]:
# ── E4: Compare selection methods on same image ──────────────────────────────
import os, glob
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

E4_BASE = '/content/drive/MyDrive/ThesisOutputs/e4_selection'
METHODS = ['mean', 'sum', 'mean_area']

for method in METHODS:
    out = f'{E4_BASE}/{method}'
    !python visualize_pipeline.py \
      --image-path "{DEBUG_IMAGE}" \
      --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
      --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
      --image-size 384 \
      --confidence-threshold 0.5 \
      --cam-percentile 85.0 \
      --max-points 5 \
      --min-component-area 100 \
      --mask-score-threshold 0.4 \
      --selection-method "{method}" \
      --output-path "{out}/pipeline_strip.png"
    print(f'Done: {method}')

# Display side-by-side comparison
stem = os.path.splitext(os.path.basename(DEBUG_IMAGE))[0]

# Load pseudo mask panel (index 5 in the strip = rightmost 6th panel)
# We compare the full strips
fig, axes = plt.subplots(len(METHODS), 1, figsize=(22, 3.5 * len(METHODS)))
for ax, method in zip(axes, METHODS):
    strip_path = f'{E4_BASE}/{method}/pipeline_strip.png'
    if os.path.exists(strip_path):
        ax.imshow(np.array(Image.open(strip_path)))
        ax.set_ylabel(method, fontsize=12, rotation=0, labelpad=60, va='center')
    else:
        ax.text(0.5, 0.5, f'{method}: not found', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

plt.suptitle(f'E4 — Selection Method Comparison: {stem}', fontsize=12)
plt.tight_layout()
plt.show()

### E5 — Batch 20 ảnh đại diện (5 per class)

Chạy `visualize_pipeline.py` trên 5 ảnh mỗi class (hand/leg/hip/shoulder). Xuất bảng đánh giá và grid ảnh để xác định bottleneck.

In [ ]:
# ── E5: Batch 20 representative images (5 per anatomy class) ─────────────────
import os, glob, csv, random
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Pick 5 images per class from the dataset CSV
DATASET_CSV = os.path.join(DATASET_PATH, 'dataset.csv')
IMAGE_DIR   = os.path.join(DATASET_PATH, 'images')
CLASSES     = ['hand', 'leg', 'hip', 'shoulder']
N_PER_CLASS = 5
E5_OUTPUT   = '/content/drive/MyDrive/ThesisOutputs/e5_batch20'

# Collect per-class image lists from CSV
per_class: dict[str, list[str]] = {c: [] for c in CLASSES}
with open(DATASET_CSV, encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames or []
    img_col = 'image_id' if 'image_id' in fieldnames else fieldnames[0]
    for row in reader:
        for cls in CLASSES:
            if cls in fieldnames and row.get(cls, '0').strip() in ('1', 'True', 'true'):
                img_path = os.path.join(IMAGE_DIR, row[img_col])
                if not os.path.exists(img_path):
                    # try recursive search
                    found = glob.glob(f'{IMAGE_DIR}/**/{row[img_col]}', recursive=True)
                    if found: img_path = found[0]
                if os.path.exists(img_path):
                    per_class[cls].append(img_path)

random.seed(42)
selected: list[tuple[str, str]] = []  # (class, path)
for cls in CLASSES:
    sample = random.sample(per_class[cls], min(N_PER_CLASS, len(per_class[cls])))
    for p in sample:
        selected.append((cls, p))
    print(f'  {cls}: {len(sample)} images selected (pool={len(per_class[cls])})')

print(f'\nRunning pipeline on {len(selected)} images...')
for cls, img_path in selected:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    out = f'{E5_OUTPUT}/{cls}'
    !python visualize_pipeline.py \
      --image-path "{img_path}" \
      --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
      --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
      --image-size 384 \
      --confidence-threshold 0.5 \
      --cam-percentile 85.0 \
      --max-points 5 \
      --min-component-area 100 \
      --mask-score-threshold 0.4 \
      --selection-method mean \
      --output-path "{out}/{stem}_pipeline.png"

print('Done. Displaying results...')

# Display grid: one row per image, grouped by class
fig_rows = []
for cls in CLASSES:
    strips = sorted(glob.glob(f'{E5_OUTPUT}/{cls}/*_pipeline.png'))
    for p in strips[:N_PER_CLASS]:
        fig_rows.append((cls, p))

if fig_rows:
    fig, axes = plt.subplots(len(fig_rows), 1, figsize=(22, 3.5 * len(fig_rows)))
    if len(fig_rows) == 1: axes = [axes]
    prev_cls = None
    for ax, (cls, path) in zip(axes, fig_rows):
        img = np.array(Image.open(path))
        ax.imshow(img)
        stem = os.path.splitext(os.path.basename(path))[0].replace('_pipeline', '')
        label = f'[{cls}] {stem}' if cls != prev_cls else f'       {stem}'
        ax.set_ylabel(label, fontsize=9, rotation=0, labelpad=120, va='center')
        ax.axis('off')
        prev_cls = cls
    plt.suptitle('E5 — Batch 20 Representative Images', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{E5_OUTPUT}/batch20_grid.png', dpi=80, bbox_inches='tight')
    plt.show()
    print(f'Grid saved to {E5_OUTPUT}/batch20_grid.png')

---
## 7. Bottleneck Analysis (100 ảnh)

> **Mục tiêu:** Trả lời câu hỏi: pipeline đang fail vì SAM hay vì Selection?
>
> - **Task A** — SAM candidate success rate: có ít nhất 1 mask bao phủ đúng xương không?
> - **Task B** — So sánh `mean` / `sum` / `mean_area` trên 100 ảnh, đếm GOOD/PARTIAL/BAD
> - **Task C** — Winner & runner-up từ `scores.json`, phân tích score gap
>
> Chạy **Task A+B+C cùng lúc** — 1 lần pass qua 100 ảnh, lưu kết quả, rồi hiển thị báo cáo.

In [ ]:
# ── Task A + B + C: Run 100-image bottleneck analysis ────────────────────────
# Chạy một lần — lưu kết quả vào ANALYSIS_OUTPUT/results/
# Mỗi ảnh: (a) sinh debug output bằng mean/sum/mean_area, (b) đọc scores.json
#
# Prerequisite: CLASSIFIER_CHECKPOINT, SAM_CHECKPOINT_PATH, DATASET_PATH đã định nghĩa.

import os, glob, csv, json, random, subprocess, sys
from pathlib import Path

ANALYSIS_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/bottleneck_100'
ANALYSIS_N      = 100   # số ảnh khảo sát
METHODS         = ['mean', 'sum', 'mean_area']
CAM_COVER_THRESHOLD = 0.15   # min CAM mean inside mask → mask "covers" xương
RANDOM_SEED     = 42

os.makedirs(ANALYSIS_OUTPUT, exist_ok=True)

# ── 1. Lấy 100 ảnh ngẫu nhiên ────────────────────────────────────────────────
DATASET_CSV = os.path.join(DATASET_PATH, 'dataset.csv')
IMAGE_DIR   = os.path.join(DATASET_PATH, 'images')

all_images = (
    glob.glob(f'{IMAGE_DIR}/**/*.jpg', recursive=True) +
    glob.glob(f'{IMAGE_DIR}/**/*.jpeg', recursive=True) +
    glob.glob(f'{IMAGE_DIR}/**/*.png', recursive=True)
)
random.seed(RANDOM_SEED)
sampled = random.sample(all_images, min(ANALYSIS_N, len(all_images)))
print(f'Sampled {len(sampled)} images from {len(all_images)} total')

# ── 2. Chạy visualize_pipeline.py --debug cho mỗi ảnh, mỗi method ────────────
# Lưu strips vào ANALYSIS_OUTPUT/<method>/<stem>_pipeline.png
# Lưu debug (scores.json) vào ANALYSIS_OUTPUT/<method>/debug/<stem>/

for i, img_path in enumerate(sampled):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    print(f'[{i+1:3d}/{len(sampled)}] {stem}', end='  ')
    for method in METHODS:
        out_dir = f'{ANALYSIS_OUTPUT}/{method}'
        strip_path = f'{out_dir}/{stem}_pipeline.png'
        if os.path.exists(strip_path):
            print(f'{method}:cached', end='  ')
            continue
        ret = subprocess.run(
            [sys.executable, 'visualize_pipeline.py',
             '--image-path', img_path,
             '--classifier-checkpoint', CLASSIFIER_CHECKPOINT,
             '--sam-checkpoint', SAM_CHECKPOINT_PATH,
             '--image-size', '384',
             '--confidence-threshold', '0.5',
             '--cam-percentile', '85.0',
             '--max-points', '5',
             '--min-component-area', '100',
             '--mask-score-threshold', '0.4',
             '--selection-method', method,
             '--debug',
             '--output-path', strip_path],
            capture_output=True, text=True
        )
        ok = '✓' if ret.returncode == 0 else '✗'
        print(f'{method}:{ok}', end='  ')
        if ret.returncode != 0:
            print(f'\n  ERROR: {ret.stderr[-300:]}', end='')
    print()

print('\nAll runs complete.')
print(f'Results in: {ANALYSIS_OUTPUT}/')

In [ ]:
# ── Task A + B + C: Analyse results ──────────────────────────────────────────
# Đọc debug outputs, tính:
#   Task A — SAM candidate success rate (YES nếu SAM iou score >= ngưỡng)
#   Task B — so sánh pseudo mask AREA theo method (proxy coverage; scores.json
#             chứa SAM's own iou scores — giống nhau với mọi selection method,
#             không dùng được. Dùng pixel area của last panel trong strip.)
#   Task C — winner / runner-up per image (SAM iou scores), score gap

import os, json, glob
import numpy as np
from PIL import Image
from collections import defaultdict

METHODS = ['mean', 'sum', 'mean_area']
SAM_SCORE_THRESHOLD = 0.7    # task A: SAM iou score >= này → candidate "good"
GOOD_AREA_FRAC      = 0.05   # task B: mask area >= 5% image → GOOD
PARTIAL_AREA_FRAC   = 0.01   # task B: >= 1% → PARTIAL, else BAD
IMAGE_PIXELS        = 384 * 384

# Collect all stems from the 'mean' run (reference)
ref_strips = sorted(glob.glob(f'{ANALYSIS_OUTPUT}/mean/*_pipeline.png'))
stems = [os.path.basename(p).replace('_pipeline.png', '') for p in ref_strips]
print(f'Analysing {len(stems)} images...\n')

task_a_results = {}
task_b_results = defaultdict(dict)
task_c_results = {}

for stem in stems:
    # ── Task A: SAM iou scores from 'mean' debug dir ─────────────────────────
    scores_path = f'{ANALYSIS_OUTPUT}/mean/debug/{stem}/scores.json'
    if not os.path.exists(scores_path):
        task_a_results[stem] = 'NO (no debug)'
        for m in METHODS:
            task_b_results[stem][m] = 'N/A'
        continue

    with open(scores_path) as f:
        scores_data = json.load(f)

    sam_scores = {k: v['score'] for k, v in scores_data.items()}
    sorted_scores = sorted(sam_scores.items(), key=lambda x: x[1], reverse=True)

    max_score = sorted_scores[0][1] if sorted_scores else 0.0
    task_a_results[stem] = 'YES' if max_score >= SAM_SCORE_THRESHOLD else 'NO'

    # Task C: winner / runner-up
    winner = sorted_scores[0] if sorted_scores else ('none', 0.0)
    runner = sorted_scores[1] if len(sorted_scores) >= 2 else ('none', 0.0)
    task_c_results[stem] = {
        'winner': winner[0], 'winner_score': winner[1],
        'runner_up': runner[0], 'runner_score': runner[1],
        'gap': winner[1] - runner[1], 'n_masks': len(sorted_scores),
    }

    # ── Task B: measure pseudo mask area from last panel of pipeline strip ───
    # scores.json stores SAM iou scores (same for all methods), not CAM scores.
    # Instead, read the rightmost 1/6 of the pipeline strip as proxy for coverage.
    for method in METHODS:
        strip_path = f'{ANALYSIS_OUTPUT}/{method}/{stem}_pipeline.png'
        if not os.path.exists(strip_path):
            task_b_results[stem][method] = 'N/A'
            continue
        strip = np.array(Image.open(strip_path).convert('L'))
        panel_w = strip.shape[1] // 6
        last_panel = strip[:, -panel_w:]
        bone_pixels = int((last_panel[22:] > 127).sum())  # skip 22px label header
        area_frac = bone_pixels / IMAGE_PIXELS
        if area_frac >= GOOD_AREA_FRAC:
            task_b_results[stem][method] = 'GOOD'
        elif area_frac >= PARTIAL_AREA_FRAC:
            task_b_results[stem][method] = 'PARTIAL'
        else:
            task_b_results[stem][method] = 'BAD'

# ── Print Task A ──────────────────────────────────────────────────────────────
yes_count = sum(1 for v in task_a_results.values() if v == 'YES')
no_count  = sum(1 for v in task_a_results.values() if v.startswith('NO'))
total     = len(task_a_results)
print('═' * 60)
print('TASK A — SAM Candidate Success Rate')
print('═' * 60)
print(f'  YES (SAM iou score >= {SAM_SCORE_THRESHOLD}): {yes_count}/{total}  ({yes_count/total*100:.1f}%)')
print(f'  NO:                                            {no_count}/{total}  ({no_count/total*100:.1f}%)')
print()

# ── Print Task B ──────────────────────────────────────────────────────────────
print('═' * 60)
print('TASK B — Pseudo Mask Coverage by Selection Method')
print(f'  GOOD >= {GOOD_AREA_FRAC*100:.0f}% px | PARTIAL >= {PARTIAL_AREA_FRAC*100:.0f}% px | BAD otherwise')
print('═' * 60)
header = f"{'stem':<30}" + ''.join(f'{m:>12}' for m in METHODS)
print(header)
print('-' * (30 + 12 * len(METHODS)))

method_counts = {m: defaultdict(int) for m in METHODS}
for stem in stems:
    row = f'{stem[:29]:<30}'
    for m in METHODS:
        rating = task_b_results[stem].get(m, 'N/A')
        row += f'{rating:>12}'
        method_counts[m][rating] += 1
    print(row)

print('-' * (30 + 12 * len(METHODS)))
summary_row = f"{'TOTAL':<30}"
for m in METHODS:
    c = method_counts[m]
    summary_row += f"G:{c['GOOD']} P:{c['PARTIAL']} B:{c['BAD']}".rjust(12)
print(summary_row)
print()

# ── Print Task C ──────────────────────────────────────────────────────────────
print('═' * 60)
print('TASK C — SAM Candidate Winner & Runner-up (by SAM iou score)')
print('═' * 60)
sorted_by_gap = sorted(task_c_results.items(), key=lambda x: x[1]['gap'], reverse=True)
print(f"{'stem':<30}  {'winner':<8}  {'score':>6}  {'runner':>8}  {'score':>6}  {'gap':>6}  {'n':>3}")
print('-' * 75)
for stem, r in sorted_by_gap[:50]:
    print(f"{stem[:29]:<30}  {r['winner']:<8}  {r['winner_score']:>6.3f}"
          f"  {r['runner_up']:>8}  {r['runner_score']:>6.3f}  {r['gap']:>6.3f}  {r['n_masks']:>3}")

results_path = f'{ANALYSIS_OUTPUT}/bottleneck_results.json'
with open(results_path, 'w') as f:
    json.dump({
        'task_a': task_a_results,
        'task_b': {s: dict(v) for s, v in task_b_results.items()},
        'task_c': task_c_results,
        'summary': {
            'sam_success_rate': yes_count / total if total else 0,
            'method_counts': {m: dict(method_counts[m]) for m in METHODS},
        }
    }, f, indent=2)
print(f'\nFull results saved to {results_path}')

---
## 8. Bước 2 — Multi-Mask Fusion

> Thay vì chọn 1 mask tốt nhất, lấy **top-k masks** và hợp lại (union hoặc intersection).
>
> - **top3_union**: `mask1 | mask2 | mask3` — bao phủ nhiều hơn, phù hợp khi CAM cover thấp
> - **top2_intersection**: `mask1 & mask2` — chính xác hơn, phù hợp khi SAM score gap lớn
>
> Cần thêm `--fusion-topk` vào `select_and_fuse_masks` trong `pseudo/mask_selection.py`.

In [ ]:
# ── Bước 2: Multi-Mask Fusion — so sánh các chiến lược kết hợp mask ──────────
# fusion_topk:
#   0  → OR của tất cả mask >= threshold (default cũ)
#   3  → union top-3 masks (covering)
#   -2 → intersection top-2 masks (precise)

import subprocess, sys, os, glob
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

FUSION_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/fusion_compare'
FUSION_IMAGE  = DEBUG_IMAGE   # same image as E1-E4

FUSION_STRATEGIES = [
    ('default_OR',          ['--selection-method', 'mean',       '--fusion-topk', '0']),
    ('top3_union_mean',     ['--selection-method', 'mean',       '--fusion-topk', '3']),
    ('top2_intersect_mean', ['--selection-method', 'mean',       '--fusion-topk', '-2']),
    ('top3_union_meanarea', ['--selection-method', 'mean_area',  '--fusion-topk', '3']),
]

os.makedirs(FUSION_OUTPUT, exist_ok=True)

for label, extra_args in FUSION_STRATEGIES:
    out_path = f'{FUSION_OUTPUT}/{label}_pipeline.png'
    if os.path.exists(out_path):
        print(f'{label}: cached')
        continue
    ret = subprocess.run(
        [sys.executable, 'visualize_pipeline.py',
         '--image-path', FUSION_IMAGE,
         '--classifier-checkpoint', CLASSIFIER_CHECKPOINT,
         '--sam-checkpoint', SAM_CHECKPOINT_PATH,
         '--image-size', '384',
         '--confidence-threshold', '0.5',
         '--cam-percentile', '85.0',
         '--max-points', '5',
         '--min-component-area', '100',
         '--mask-score-threshold', '0.4',
         '--output-path', out_path] + extra_args,
        capture_output=True, text=True
    )
    if ret.returncode == 0:
        print(f'{label}: ✓')
    else:
        print(f'{label}: ✗')
        print(ret.stderr[-300:])

# Display
fig, axes = plt.subplots(len(FUSION_STRATEGIES), 1, figsize=(22, 3.5 * len(FUSION_STRATEGIES)))
if len(FUSION_STRATEGIES) == 1:
    axes = [axes]
for ax, (label, _) in zip(axes, FUSION_STRATEGIES):
    p = f'{FUSION_OUTPUT}/{label}_pipeline.png'
    if os.path.exists(p):
        ax.imshow(np.array(Image.open(p)))
        ax.set_ylabel(label, fontsize=10, rotation=0, labelpad=130, va='center')
    else:
        ax.text(0.5, 0.5, f'{label}: not found', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')
plt.suptitle('Bước 2 — Fusion Strategy Comparison', fontsize=12)
plt.tight_layout()
plt.show()

---
## 9. Bước 3 — Train U-Net với Best Strategy

> Sau khi có kết quả Task A/B, chọn `BEST_METHOD` và `BEST_FUSION_TOPK` tốt nhất,
> regen pseudo masks cho toàn bộ dataset, train U-Net, so sánh Dice/IoU.

In [ ]:
# ── Bước 3: Regenerate pseudo masks với best strategy, train U-Net ────────────
# Sau khi phân tích Task A/B, điều chỉnh 2 dòng này:
BEST_METHOD      = 'mean'      # 'mean' | 'sum' | 'mean_area'
BEST_FUSION_TOPK = 0           # 0=OR-all, 3=union-top3, -2=intersect-top2

BEST_MASKS_OUTPUT = f'/content/drive/MyDrive/ThesisOutputs/pseudo_masks_best_{BEST_METHOD}_topk{BEST_FUSION_TOPK}'
BEST_SEG_OUTPUT   = f'/content/drive/MyDrive/ThesisOutputs/segmentation_best_{BEST_METHOD}_topk{BEST_FUSION_TOPK}'

print(f'Strategy: selection={BEST_METHOD}, fusion_topk={BEST_FUSION_TOPK}')
print(f'Pseudo masks → {BEST_MASKS_OUTPUT}')
print(f'U-Net output → {BEST_SEG_OUTPUT}')

# ── Step 3a: Generate pseudo masks ───────────────────────────────────────────
!python generate_pseudo_masks.py \
  --data-root "{DATASET_PATH}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --target-columns hand,leg,hip,shoulder \
  --image-size 384 \
  --batch-size 1 \
  --num-workers 2 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --selection-method "{BEST_METHOD}" \
  --closing-kernel 5 \
  --opening-kernel 3 \
  --min-size 200 \
  --output-dir "{BEST_MASKS_OUTPUT}"

In [ ]:
# ── Step 3b: Train U-Net với best pseudo masks ────────────────────────────────
!python train_segmentation.py \
  --data-root "{DATASET_PATH}" \
  --mask-root "{BEST_MASKS_OUTPUT}/masks" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 2 \
  --epochs 25 \
  --output-dir "{BEST_SEG_OUTPUT}"

# ── Step 3c: Compare training curves: original vs best strategy ───────────────
import pandas as pd
import matplotlib.pyplot as plt
import os

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

runs = [
    ('Original (mean, OR-all)', SEGMENTATION_OUTPUT),
    (f'Best ({BEST_METHOD}, topk={BEST_FUSION_TOPK})', BEST_SEG_OUTPUT),
]

colors = ['steelblue', 'tomato']
for metric, ax in zip(['loss', 'dice', 'iou'], axes):
    for (label, out_dir), color in zip(runs, colors):
        log = f'{out_dir}/training_log.csv'
        if os.path.exists(log):
            df = pd.read_csv(log)
            ax.plot(df['epoch'], df[f'val_{metric}'], label=label, color=color)
    ax.set_title(f'Val {metric.capitalize()}')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=7)

plt.suptitle('Bước 3 — U-Net: Original vs Best Strategy', fontsize=12)
plt.tight_layout()
plt.show()

# ── Print summary table ────────────────────────────────────────────────────────
print(f'\n{"Run":<40}  {"Val Dice":>9}  {"Val IoU":>9}')
print('-' * 62)
for label, out_dir in runs:
    log = f'{out_dir}/training_log.csv'
    if os.path.exists(log):
        df = pd.read_csv(log)
        best = df.loc[df['val_dice'].idxmax()]
        print(f'{label:<40}  {best["val_dice"]:>9.4f}  {best["val_iou"]:>9.4f}')
    else:
        print(f'{label:<40}  {"N/A":>9}  {"N/A":>9}')

---
## 10. Kiểm chứng giả thuyết — Ranking chọn sai?

> **Giả thuyết:** SAM sinh được mask xương tốt, nhưng scoring chọn sai mask.
>
> | Giai đoạn | Mục tiêu |
> |-----------|---------|
> | **GĐ1** | Export CSV toàn bộ candidates → label thủ công → tính Top-1/3/5 accuracy |
> | **GĐ2** | Benchmark 5 scoring methods trên cùng candidates có label |
> | **GĐ3** | Tìm scoring method tốt nhất |
> | **GĐ4** | Benchmark fusion strategies với scoring tốt nhất |
> | **GĐ5** | Bảng tổng kết: method × fusion → bone coverage |
> | **GĐ6** | Train U-Net với 2–3 chiến lược tốt nhất, so sánh Dice/IoU |
>
> **Prerequisite:** Đã chạy Section 7 (bottleneck_100 debug outputs tồn tại).

### Giai đoạn 1 — Export candidates CSV + Label thủ công

In [ ]:
# ── GĐ1a: Export candidates CSV ──────────────────────────────────────────────
# Đọc scores.json từ debug outputs, export ra CSV để label thủ công.
# Mỗi row = 1 candidate mask: image_id, mask_id, sam_score, area, rank

import os, json, glob, csv
import numpy as np

LABEL_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/gd1_labeling'
os.makedirs(LABEL_OUTPUT, exist_ok=True)

# Dùng lại bottleneck_100 debug dirs (Section 7)
DEBUG_BASE = f'{ANALYSIS_OUTPUT}/mean'   # debug từ mean run

rows = []
stems_found = []

mask_dirs = sorted(glob.glob(f'{DEBUG_BASE}/debug/*/scores.json'))
for scores_path in mask_dirs:
    stem = os.path.basename(os.path.dirname(scores_path))
    with open(scores_path) as f:
        data = json.load(f)

    # sort by SAM iou score to get rank
    sorted_masks = sorted(data.items(), key=lambda x: x[1]['score'], reverse=True)
    for rank, (mask_id, info) in enumerate(sorted_masks, start=1):
        rows.append({
            'image_id':  stem,
            'mask_id':   mask_id,
            'sam_score': round(info['score'], 4),
            'area':      info['area'],
            'rank':      rank,
            'label':     '',   # để trống — điền thủ công: GOOD_BONE / PARTIAL_BONE / BAD
        })
    stems_found.append(stem)

csv_path = f'{LABEL_OUTPUT}/candidates_to_label.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['image_id','mask_id','sam_score','area','rank','label'])
    writer.writeheader()
    writer.writerows(rows)

print(f'Exported {len(rows)} candidates from {len(stems_found)} images')
print(f'CSV saved: {csv_path}')
print()
print('Bước tiếp theo:')
print('  1. Tải file CSV về máy')
print('  2. Mở mask PNG tương ứng tại:')
print(f'       {DEBUG_BASE}/debug/<image_id>/mask_<n>.png')
print('  3. Điền cột "label": GOOD_BONE | PARTIAL_BONE | BAD')
print('  4. Upload lại CSV đã điền lên Drive cùng thư mục')

In [ ]:
# ── GĐ1b: Hiển thị ảnh để label thủ công ─────────────────────────────────────
# Xem overlay của từng candidate mask để dễ label.
# Thay IMAGE_ID bằng stem muốn xem.

import os, json, glob
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

IMAGE_ID  = stems_found[0]   # đổi thành bất kỳ stem nào trong stems_found
DEBUG_DIR = f'{DEBUG_BASE}/debug/{IMAGE_ID}'

mask_files    = sorted(glob.glob(f'{DEBUG_DIR}/mask_*.png'))
overlay_files = sorted(glob.glob(f'{DEBUG_DIR}/overlay_mask_*.png'))
scores_path   = f'{DEBUG_DIR}/scores.json'

if not mask_files:
    print(f'No masks found for {IMAGE_ID}. Run Section 7 first.')
else:
    with open(scores_path) as f:
        scores_data = json.load(f)

    n = len(mask_files)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    sorted_by_sam = sorted(scores_data.items(), key=lambda x: x[1]['score'], reverse=True)
    rank_map = {k: r+1 for r, (k, _) in enumerate(sorted_by_sam)}

    for i, mf in enumerate(mask_files):
        mask_id = f"mask_{i}"
        info    = scores_data.get(mask_id, {})
        rank    = rank_map.get(mask_id, '?')
        title   = f"{mask_id}\nsam={info.get('score',0):.3f} area={info.get('area',0)}\nrank={rank}"

        axes[0][i].imshow(np.array(Image.open(mf)), cmap='gray')
        axes[0][i].set_title(title, fontsize=8)
        axes[0][i].axis('off')

        of = overlay_files[i] if i < len(overlay_files) else None
        if of and os.path.exists(of):
            axes[1][i].imshow(np.array(Image.open(of).convert('RGB')))
        axes[1][i].set_title('overlay', fontsize=8)
        axes[1][i].axis('off')

    plt.suptitle(f'GĐ1 — Label candidates: {IMAGE_ID}\n'
                 f'Label mỗi mask: GOOD_BONE / PARTIAL_BONE / BAD', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── GĐ1c: Tính Top-1 / Top-3 / Top-5 accuracy sau khi đã label ───────────────
# Chạy sau khi đã upload CSV đã điền vào Drive.
# Kết quả trả lời: "SAM sinh GOOD_BONE mask, nhưng nó đứng rank bao nhiêu?"

import csv
from collections import defaultdict

LABELED_CSV = f'{LABEL_OUTPUT}/candidates_to_label.csv'

if not os.path.exists(LABELED_CSV):
    print(f'Chưa có file label: {LABELED_CSV}')
    print('Điền cột "label" trong CSV rồi upload lên Drive.')
else:
    # Load labeled data
    per_image = defaultdict(list)   # image_id → list of (rank, label)
    with open(LABELED_CSV, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['label'].strip():
                per_image[row['image_id']].append((int(row['rank']), row['label'].strip().upper()))

    n_images = len(per_image)
    top1_good = top3_good = top5_good = 0
    sam_has_good = 0   # images where GOOD_BONE exists at all

    for image_id, mask_labels in per_image.items():
        # sorted by rank
        by_rank = sorted(mask_labels, key=lambda x: x[0])
        good_ranks = [r for r, l in by_rank if l == 'GOOD_BONE']

        if good_ranks:
            sam_has_good += 1
            min_good_rank = min(good_ranks)
            if min_good_rank == 1: top1_good += 1
            if min_good_rank <= 3: top3_good += 1
            if min_good_rank <= 5: top5_good += 1

    print('═' * 55)
    print('GĐ1 — SAM Candidate Accuracy (thủ công)')
    print('═' * 55)
    print(f'  Tổng ảnh đã label:          {n_images}')
    print(f'  SAM có GOOD_BONE candidate:  {sam_has_good}/{n_images}  ({sam_has_good/n_images*100:.1f}%)')
    print()
    print(f'  Trong số ảnh có GOOD_BONE ({sam_has_good}):')
    if sam_has_good:
        print(f'    Top-1 accuracy:  {top1_good}/{sam_has_good}  ({top1_good/sam_has_good*100:.1f}%)')
        print(f'    Top-3 accuracy:  {top3_good}/{sam_has_good}  ({top3_good/sam_has_good*100:.1f}%)')
        print(f'    Top-5 accuracy:  {top5_good}/{sam_has_good}  ({top5_good/sam_has_good*100:.1f}%)')
    print()
    if top1_good / max(sam_has_good, 1) < 0.5 and top3_good / max(sam_has_good, 1) > 0.7:
        print('  ✓ KẾT LUẬN: SAM tốt, Ranking tệ → benchmark scoring methods (GĐ2)')
    elif sam_has_good / max(n_images, 1) < 0.6:
        print('  ✗ KẾT LUẬN: SAM không sinh được mask tốt → cần nhiều prompts hoặc SAM2')
    else:
        print('  → Xem thêm GĐ2 để xác định scoring method tốt nhất')

### Giai đoạn 2 & 3 — Benchmark 5 scoring methods

In [ ]:
# ── GĐ2+3: Benchmark 5 scoring methods trên labeled candidates ───────────────
# Với mỗi ảnh đã label, tính CAM score theo 5 methods,
# xem GOOD_BONE mask được rank bao nhiêu.
# → Tìm method nào đặt GOOD_BONE lên Top-1 nhiều nhất.

import os, json, glob, csv, sys
import numpy as np
from PIL import Image
from collections import defaultdict

sys.path.insert(0, '/content/Thesis/project')
from pseudo.mask_selection import score_masks

SCORING_METHODS = ['mean', 'sum', 'mean_area', 'coverage', 'hybrid']
DEBUG_BASE      = f'{ANALYSIS_OUTPUT}/mean'
LABELED_CSV     = f'{LABEL_OUTPUT}/candidates_to_label.csv'

if not os.path.exists(LABELED_CSV):
    print('Chưa có file label. Chạy GĐ1a → label → upload CSV.')
else:
    # Load labels
    labels = {}   # (image_id, mask_id) → label
    with open(LABELED_CSV, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['label'].strip():
                labels[(row['image_id'], row['mask_id'])] = row['label'].strip().upper()

    labeled_images = list({img for img, _ in labels})
    print(f'Loaded labels for {len(labeled_images)} images, {len(labels)} masks\n')

    # Per-method: count images where GOOD_BONE is rank-1, rank-top3, rank-top5
    method_top1  = defaultdict(int)
    method_top3  = defaultdict(int)
    method_top5  = defaultdict(int)
    method_total = 0   # images that have at least one GOOD_BONE

    for image_id in labeled_images:
        debug_dir   = f'{DEBUG_BASE}/debug/{image_id}'
        scores_path = f'{debug_dir}/scores.json'
        if not os.path.exists(scores_path):
            continue

        with open(scores_path) as f:
            scores_data = json.load(f)

        # Load mask PNGs to numpy
        mask_ids   = sorted(scores_data.keys())
        mask_array = []
        for mid in mask_ids:
            mp = f'{debug_dir}/{mid}.png'
            if os.path.exists(mp):
                m = np.array(Image.open(mp).convert('L')) > 127
                mask_array.append(m)

        if not mask_array:
            continue
        masks = np.stack(mask_array, axis=0).astype(np.uint8)  # [N, H, W]

        # Load fused CAM from pipeline strip (second panel = LayerCAM overlay)
        # We need the raw fused_cam, but it wasn't saved separately.
        # Proxy: use the cam overlay image (grayscale) as bone_cam approximation.
        cam_overlay_path = glob.glob(f'{ANALYSIS_OUTPUT}/mean/*_{image_id}*layercam*')
        cam_overlay_path = cam_overlay_path[0] if cam_overlay_path else None

        # Better: read from overlay dir of pseudo_masks if available
        overlay_candidates = (
            glob.glob(f'{PSEUDO_MASK_OUTPUT}/overlays/{image_id}_fused_layercam.png') +
            (glob.glob(cam_overlay_path) if cam_overlay_path else [])
        )
        if not overlay_candidates:
            continue

        cam_img   = np.array(Image.open(overlay_candidates[0]).convert('L'), dtype=np.float32)
        bone_cam  = cam_img / 255.0   # normalize to [0,1]
        # Resize if needed
        if bone_cam.shape != masks.shape[1:]:
            from PIL import Image as PILImage
            cam_pil  = PILImage.fromarray((bone_cam * 255).astype(np.uint8))
            cam_pil  = cam_pil.resize((masks.shape[2], masks.shape[1]), PILImage.BILINEAR)
            bone_cam = np.array(cam_pil, dtype=np.float32) / 255.0

        # Get GT good rank for this image
        good_mask_ids = [mid for mid in mask_ids
                         if labels.get((image_id, mid)) == 'GOOD_BONE']
        if not good_mask_ids:
            continue

        method_total += 1

        for method in SCORING_METHODS:
            cam_scores = score_masks(masks, bone_cam, method=method)
            order      = np.argsort(cam_scores)[::-1]
            rank_map   = {mask_ids[i]: int(r)+1 for r, i in enumerate(order)}

            best_good_rank = min(rank_map.get(mid, 999) for mid in good_mask_ids)
            if best_good_rank == 1: method_top1[method]  += 1
            if best_good_rank <= 3: method_top3[method]  += 1
            if best_good_rank <= 5: method_top5[method]  += 1

    # ── Print benchmark table ─────────────────────────────────────────────────
    print('═' * 65)
    print(f'GĐ2+3 — Scoring Method Benchmark  (n={method_total} images with GOOD_BONE)')
    print('═' * 65)
    print(f"{'Method':<12}  {'Top-1':>8}  {'Top-1%':>7}  {'Top-3':>8}  {'Top-3%':>7}  {'Top-5':>8}  {'Top-5%':>7}")
    print('-' * 65)
    best_method = None
    best_top3   = -1
    for m in SCORING_METHODS:
        t1 = method_top1[m]; t3 = method_top3[m]; t5 = method_top5[m]
        n  = max(method_total, 1)
        print(f"{m:<12}  {t1:>8}  {t1/n*100:>6.1f}%  {t3:>8}  {t3/n*100:>6.1f}%  {t5:>8}  {t5/n*100:>6.1f}%")
        if t3 > best_top3:
            best_top3   = t3
            best_method = m
    print('=' * 65)
    print(f'\n✓ Best scoring method (Top-3): {best_method}')
    print(f'  → Dùng --selection-method {best_method} cho GĐ4 trở đi')

### Giai đoạn 4 & 5 — Benchmark fusion strategies + Bảng tổng kết

In [ ]:
# ── GĐ4+5: Benchmark tất cả combinations method × fusion ─────────────────────
# Với mỗi ảnh đã label, tính bone_coverage của pseudo mask theo
# mọi combination (5 methods × 5 fusion strategies).
# bone_coverage = pixel overlap của pseudo_mask với GOOD_BONE masks / GOOD_BONE area.

import os, json, glob, csv, sys
import numpy as np
from PIL import Image
from collections import defaultdict

sys.path.insert(0, '/content/Thesis/project')
from pseudo.mask_selection import score_masks, select_and_fuse_masks

SCORING_METHODS  = ['mean', 'sum', 'mean_area', 'coverage', 'hybrid']
FUSION_STRATEGIES = [
    ('top1',    {'fusion_topk': 1}),
    ('top3U',   {'fusion_topk': 3}),
    ('top5U',   {'fusion_topk': 5}),
    ('OR_all',  {'fusion_topk': 0}),
    ('top2I',   {'fusion_topk': -2}),
]
LABELED_CSV = f'{LABEL_OUTPUT}/candidates_to_label.csv'
DEBUG_BASE  = f'{ANALYSIS_OUTPUT}/mean'

if not os.path.exists(LABELED_CSV):
    print('Chưa có file label.')
else:
    # Load labels
    labels = {}
    with open(LABELED_CSV, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['label'].strip():
                labels[(row['image_id'], row['mask_id'])] = row['label'].strip().upper()

    labeled_images = list({img for img, _ in labels})

    # coverage[method][fusion] = list of per-image bone_coverage (0..1)
    coverage = {m: {fs: [] for fs, _ in FUSION_STRATEGIES} for m in SCORING_METHODS}

    for image_id in labeled_images:
        debug_dir   = f'{DEBUG_BASE}/debug/{image_id}'
        scores_path = f'{debug_dir}/scores.json'
        if not os.path.exists(scores_path):
            continue

        with open(scores_path) as f:
            scores_data = json.load(f)

        mask_ids   = sorted(scores_data.keys())
        mask_array = []
        for mid in mask_ids:
            mp = f'{debug_dir}/{mid}.png'
            if os.path.exists(mp):
                mask_array.append(np.array(Image.open(mp).convert('L')) > 127)
        if not mask_array:
            continue
        masks = np.stack(mask_array, axis=0).astype(np.uint8)

        # Load CAM overlay as proxy
        overlay_candidates = glob.glob(
            f'{PSEUDO_MASK_OUTPUT}/overlays/{image_id}_fused_layercam.png')
        if not overlay_candidates:
            continue
        cam_img  = np.array(Image.open(overlay_candidates[0]).convert('L'), dtype=np.float32)
        bone_cam = cam_img / 255.0
        if bone_cam.shape != masks.shape[1:]:
            from PIL import Image as PILImg
            c = PILImg.fromarray((bone_cam*255).astype(np.uint8))
            c = c.resize((masks.shape[2], masks.shape[1]), PILImg.BILINEAR)
            bone_cam = np.array(c, dtype=np.float32) / 255.0

        # Build GT bone mask from GOOD_BONE labels
        good_mask_ids = [mid for mid in mask_ids
                         if labels.get((image_id, mid)) == 'GOOD_BONE']
        if not good_mask_ids:
            continue
        good_indices = [mask_ids.index(mid) for mid in good_mask_ids
                        if mid in mask_ids]
        gt_bone = masks[good_indices].any(axis=0).astype(bool)
        gt_area = int(gt_bone.sum())
        if gt_area == 0:
            continue

        for method in SCORING_METHODS:
            for fs_label, fs_kwargs in FUSION_STRATEGIES:
                pseudo = select_and_fuse_masks(
                    masks, bone_cam,
                    mask_score_threshold=0.0,   # no threshold — rank-based selection
                    selection_method=method,
                    **fs_kwargs,
                ).astype(bool)
                overlap   = int((pseudo & gt_bone).sum())
                cov_score = overlap / gt_area
                coverage[method][fs_label].append(cov_score)

    # ── Print summary table ───────────────────────────────────────────────────
    fs_labels = [fs for fs, _ in FUSION_STRATEGIES]
    col_w = 9

    print('═' * (14 + col_w * len(fs_labels)))
    print('GĐ4+5 — Strategy Benchmark: mean bone_coverage')
    print(f"  (coverage = overlap(pseudo, GOOD_BONE) / GOOD_BONE area)")
    print('═' * (14 + col_w * len(fs_labels)))
    header = f"{'method':<12}  " + ''.join(f'{fs:>{col_w}}' for fs in fs_labels)
    print(header)
    print('-' * (14 + col_w * len(fs_labels)))

    best_combo = (None, None, -1.0)
    for method in SCORING_METHODS:
        row = f'{method:<12}  '
        for fs_label in fs_labels:
            vals = coverage[method][fs_label]
            mean_cov = float(np.mean(vals)) if vals else 0.0
            row += f'{mean_cov*100:>{col_w-1}.1f}%'
            if mean_cov > best_combo[2]:
                best_combo = (method, fs_label, mean_cov)
        print(row)

    print('=' * (14 + col_w * len(fs_labels)))
    print(f'\n✓ Best combination: {best_combo[0]} + {best_combo[1]}  '
          f'(coverage={best_combo[2]*100:.1f}%)')
    print(f'  → Dùng cho GĐ6: --selection-method {best_combo[0]} --fusion-topk ...')

### Giai đoạn 6 — Train U-Net với Top-2/3 chiến lược tốt nhất

In [ ]:
# ── GĐ6: Train U-Net với top-2/3 chiến lược tốt nhất, so sánh Dice/IoU ────────
# Cập nhật list này dựa trên kết quả GĐ4+5:
GD6_STRATEGIES = [
    # (label,          selection_method,  fusion_topk)
    ('mean_top3U',     'mean',            3),   # chiến lược hiện tại pipeline
    ('hybrid_top3U',   'hybrid',          3),
    ('coverage_top1',  'coverage',        1),
]

import os, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

GD6_BASE = '/content/drive/MyDrive/ThesisOutputs/gd6_unet'

for label, method, topk in GD6_STRATEGIES:
    masks_out = f'{GD6_BASE}/{label}/pseudo_masks'
    seg_out   = f'{GD6_BASE}/{label}/segmentation'

    print(f'\n── {label} ──────────────────────────────────────')

    # Step a: Generate pseudo masks
    ret = subprocess.run(
        [sys.executable, 'generate_pseudo_masks.py',
         '--data-root', DATASET_PATH,
         '--classifier-checkpoint', CLASSIFIER_CHECKPOINT,
         '--sam-checkpoint', SAM_CHECKPOINT_PATH,
         '--target-columns', 'hand,leg,hip,shoulder',
         '--image-size', '384',
         '--batch-size', '1',
         '--num-workers', '2',
         '--confidence-threshold', '0.5',
         '--cam-percentile', '85.0',
         '--max-points', '5',
         '--min-component-area', '100',
         '--mask-score-threshold', '0.0',
         '--selection-method', method,
         '--fusion-topk', str(topk),
         '--closing-kernel', '5',
         '--opening-kernel', '3',
         '--min-size', '200',
         '--output-dir', masks_out],
        capture_output=True, text=True
    )
    if ret.returncode != 0:
        print(f'  mask gen FAILED: {ret.stderr[-200:]}')
        continue
    print(f'  Pseudo masks: ✓')

    # Step b: Train U-Net
    ret = subprocess.run(
        [sys.executable, 'train_segmentation.py',
         '--data-root', DATASET_PATH,
         '--mask-root', f'{masks_out}/masks',
         '--image-size', '384',
         '--batch-size', '4',
         '--num-workers', '2',
         '--epochs', '25',
         '--output-dir', seg_out],
        capture_output=True, text=True
    )
    if ret.returncode != 0:
        print(f'  U-Net train FAILED: {ret.stderr[-200:]}')
    else:
        print(f'  U-Net train: ✓')

# ── Compare results ────────────────────────────────────────────────────────────
print('\n\n' + '═' * 60)
print('GĐ6 — Final Comparison: Val Dice / Val IoU')
print('═' * 60)
print(f"{'Strategy':<22}  {'Val Dice':>9}  {'Val IoU':>9}  {'Best Epoch':>11}")
print('-' * 60)

# Include baseline (Section 4) for reference
all_runs = [('baseline_mean_OR', SEGMENTATION_OUTPUT)] + \
           [(label, f'{GD6_BASE}/{label}/segmentation') for label, _, _ in GD6_STRATEGIES]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#888888', '#e74c3c', '#2ecc71', '#3498db', '#f39c12']

for (label, out_dir), color in zip(all_runs, colors):
    log = f'{out_dir}/training_log.csv'
    if not os.path.exists(log):
        print(f'{label:<22}  {"N/A":>9}  {"N/A":>9}  {"N/A":>11}')
        continue
    df   = pd.read_csv(log)
    best = df.loc[df['val_dice'].idxmax()]
    print(f'{label:<22}  {best["val_dice"]:>9.4f}  {best["val_iou"]:>9.4f}  {int(best["epoch"]):>11}')
    axes[0].plot(df['epoch'], df['val_dice'], label=label, color=color)
    axes[1].plot(df['epoch'], df['val_iou'],  label=label, color=color)

for ax, title in zip(axes, ['Val Dice', 'Val IoU']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle('GĐ6 — U-Net: Strategy Comparison', fontsize=12)
plt.tight_layout()
plt.savefig(f'{GD6_BASE}/gd6_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'\nPlot saved: {GD6_BASE}/gd6_comparison.png')